# Project_R — one yellow page

**Runtime → Run all.** The working prints collapse into one Rapido-yellow page (KPI cards + two sliders), same maths as `./run_all.sh`.

Yellow cards move with the sliders. White cards stay locked (C1b, approval %, airport).


In [ ]:
# One cell on purpose: Jupyter/Colab stacks every cell as a separate box.
from __future__ import annotations

import contextlib
import io
import os
import runpy
import subprocess
import sys
from pathlib import Path

from IPython.display import clear_output, display, HTML

ROOT = Path(".").resolve()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

display(HTML(
    "<p style='font-family:sans-serif'>Installing packages and running the working… "
    "this is the only status line. The yellow page appears when it finishes.</p>"
))

req = ROOT / "requirements.txt"
if req.exists():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])

STEPS = [
    ("01_data_audit.py", "Audit"),
    ("02_funnel.py", "Funnel"),
    ("03_dropoff.py", "Drop-off"),
    ("04_channel_leaks.py", "Channel"),
    ("05_campaign.py", "Campaign"),
    ("06_airport_hourly.py", "Airport hours"),
    ("07_airport_trips.py", "Airport trips"),
    ("08_intervention_sizing.py", "Sizing"),
    ("build_memo.py", "Memo"),
]
logs: dict[str, str] = {}
for script, label in STEPS:
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        runpy.run_path(str(ROOT / script), run_name="__main__")
    logs[label] = buf.getvalue()

import importlib
import notebook_app
import check_regression
importlib.reload(notebook_app)
from metrics import (
    C1A_CENTRAL_SHOWUP,
    airport_hourly_snapshot,
    ara_economics,
    ara_monthly_cost,
    c1a_monthly,
    c1b_monthly,
    event_funnel,
    load_onboarding,
    mature_months,
)

reg = io.StringIO()
with contextlib.redirect_stdout(reg):
    rc = check_regression.main()
if rc != 0:
    print(reg.getvalue())
    raise SystemExit("Headline regression failed")

hourly = airport_hourly_snapshot()
eco = ara_economics()
months, *_ = mature_months()
funnel = event_funnel()
n = len(funnel)
n_appr = int(funnel["final_status"].eq("approved").sum())
captains, *_ = load_onboarding()

clear_output(wait=True)
try:
    from google.colab import output as colab_output
    colab_output.no_vertical_scroll()
except Exception:
    pass

notebook_app.show_app(
    n=n,
    n_appr=n_appr,
    months=months,
    c1a_60=c1a_monthly(C1A_CENTRAL_SHOWUP),
    c1a_at_1=c1a_monthly(1.0),
    c1b=c1b_monthly(2),
    airport_unf=hourly["airport_unf_share"],
    other_unf=hourly["other_unf_share"],
    night_share=hourly["night_share_of_airport_unf"],
    cap_night=hourly["mean_captains_worst"],
    cap_day=hourly["mean_captains_rest"],
    ara_leg=eco["payout_per_eligible_leg"],
    ara_mo=ara_monthly_cost(1.0),
    mix=captains["vehicle_type"].value_counts().to_dict(),
    logs=logs,
)
